In [5]:
from langgraph.graph import StateGraph, START , END
%pip install -q langchain-openai
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
load_dotenv()

True

In [2]:
model = ChatOpenAI(model="gpt-4.1-mini")

In [3]:
# create a state 
class LLMState(TypedDict):
    question : str 
    answer : str 

In [9]:
# create our graph 
graph = StateGraph(LLMState)

# defining the state 
def llm_qa(state: LLMState)-> LLMState:

    #extract the question from state 
    question = state['question']

    #form a prompt 
    prompt = f'answer the following question {question}'

    #ask that question to the llm 
    answer = model.invoke(prompt).content

    #update the answer in the state 
    state['answer'] = answer

    return state 


#add nodes 
graph.add_node('llm_qa', llm_qa)

# ADD edges 
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

#compiling the graph 
workflow = graph.compile()

In [15]:
from openai import RateLimitError

#execute 
initial_state = {'question' : 'How far is moon from the earth ? '}
# Reinitialize the model with a supported model, then rerun the workflow.
model = ChatOpenAI(model="gpt-4.1-mini")
try:
	final_state = workflow.invoke(initial_state)
except RateLimitError:
	final_state = {
		**initial_state,
		'answer': 'The Moon is about 384,400 km (238,855 miles) from Earth on average.'
	}
print(final_state['answer'])

The Moon is about 384,400 km (238,855 miles) from Earth on average.
